# ROGII LightGBM V1 3M

This notebook trains the current best local pipeline. On Kaggle it writes `/kaggle/working/submission.csv`; locally it reads `data/raw` and writes `submissions/submission.csv`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

def find_kaggle_raw_dir():
    input_root = Path('/kaggle/input')
    candidates = [input_root / 'rogii-wellbore-geology-prediction']
    candidates.extend(sorted(path.parent for path in input_root.rglob('sample_submission.csv')))
    candidates.extend(sorted(input_root.glob('*')))
    for candidate in candidates:
        if (
            candidate.exists()
            and (candidate / 'train').exists()
            and (candidate / 'test').exists()
            and (candidate / 'sample_submission.csv').exists()
        ):
            return candidate
    available = sorted(str(path) for path in input_root.glob('*')) if input_root.exists() else []
    raise FileNotFoundError(f'Could not find Kaggle raw data under /kaggle/input. Available: {available}')

if Path('/kaggle/working').exists():
    RAW_DIR = find_kaggle_raw_dir()
    OUTPUT_PATH = Path('/kaggle/working/submission.csv')
else:
    RAW_DIR = Path('../data/raw') if Path('../data/raw').exists() else Path('data/raw')
    OUTPUT_PATH = Path('../submissions/submission.csv') if Path('../submissions').exists() else Path('submissions/submission.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Using RAW_DIR={RAW_DIR.resolve()}')
print(f'Using OUTPUT_PATH={OUTPUT_PATH.resolve()}')
RANDOM_STATE = 42
VALID_SIZE = 0.12
MAX_TRAIN_ROWS = 3_000_000
CHUNKSIZE = 250_000

FEATURE_COLUMNS = [
    'row_index', 'n_rows', 'row_from_ps', 'frac_from_ps', 'MD', 'md_from_ps',
    'X', 'Y', 'Z', 'x_from_ps', 'y_from_ps', 'z_from_ps', 'xy_dist_from_ps',
    'gr', 'gr_was_missing', 'gr_from_ps', 'gr_roll_mean_11', 'gr_roll_mean_51',
    'gr_roll_std_51', 'gr_delta_1', 'gr_delta_10', 'last_tvt_input',
    'first_tvt_input', 'tvt_input_range', 'tvt_slope_last_25',
    'tvt_slope_last_100', 'baseline_tvt', 'typewell_tvt_min',
    'typewell_tvt_max', 'typewell_gr_at_baseline_tvt',
    'gr_minus_typewell_gr_at_baseline', 'nearest_typewell_tvt_by_gr',
    'nearest_typewell_gr_diff',
]
META_COLUMNS = ['well_id', 'row_index']
TARGET_COLUMN = 'target_tvt'

def interpolate_with_extrapolation(x, y, x_query):
    valid = np.isfinite(x) & np.isfinite(y)
    x_known = x[valid]
    y_known = y[valid]
    if len(x_known) == 0:
        return np.full_like(x_query, np.nan, dtype=float)
    if len(x_known) == 1:
        return np.full_like(x_query, y_known[0], dtype=float)
    order = np.argsort(x_known)
    x_known = x_known[order]
    y_known = y_known[order]
    pred = np.interp(x_query, x_known, y_known)
    left = x_query < x_known[0]
    if left.any():
        slope = (y_known[1] - y_known[0]) / (x_known[1] - x_known[0])
        pred[left] = y_known[0] + slope * (x_query[left] - x_known[0])
    right = x_query > x_known[-1]
    if right.any():
        slope = (y_known[-1] - y_known[-2]) / (x_known[-1] - x_known[-2])
        pred[right] = y_known[-1] + slope * (x_query[right] - x_known[-1])
    return pred

def slope_last(values, x, n):
    known = values.notna()
    if known.sum() < 2:
        return 0.0
    y = values.loc[known].tail(n).to_numpy(dtype=float)
    x_tail = x.loc[known].tail(n).to_numpy(dtype=float)
    if len(y) < 2 or np.isclose(x_tail[-1], x_tail[0]):
        return 0.0
    return float((y[-1] - y[0]) / (x_tail[-1] - x_tail[0]))

def nearest_typewell_by_gr(typewell, gr_values):
    type_gr = typewell['GR'].to_numpy(dtype=float)
    type_tvt = typewell['TVT'].to_numpy(dtype=float)
    nearest_tvt = np.empty(len(gr_values), dtype=float)
    nearest_diff = np.empty(len(gr_values), dtype=float)
    valid_type = np.isfinite(type_gr)
    type_gr = type_gr[valid_type]
    type_tvt = type_tvt[valid_type]
    for i, gr in enumerate(gr_values):
        if not np.isfinite(gr) or len(type_gr) == 0:
            nearest_tvt[i] = np.nan
            nearest_diff[i] = np.nan
            continue
        idx = int(np.argmin(np.abs(type_gr - gr)))
        nearest_tvt[i] = type_tvt[idx]
        nearest_diff[i] = abs(type_gr[idx] - gr)
    return nearest_tvt, nearest_diff

def build_well_features(horizontal_path, typewell_path, split):
    well_id = horizontal_path.name.split('__')[0]
    horizontal = pd.read_csv(horizontal_path)
    typewell = pd.read_csv(typewell_path)
    row_index = np.arange(len(horizontal))
    tvt_input_missing = horizontal['TVT_input'].isna()
    ps_row = int(tvt_input_missing.idxmax()) if tvt_input_missing.any() else len(horizontal)
    ps_row_safe = min(ps_row, len(horizontal) - 1)
    gr_raw = horizontal['GR']
    gr = gr_raw.interpolate(limit_direction='both')
    md = horizontal['MD']
    baseline_tvt = interpolate_with_extrapolation(
        md.to_numpy(dtype=float),
        horizontal['TVT_input'].to_numpy(dtype=float),
        md.to_numpy(dtype=float),
    )
    typewell_sorted = typewell.sort_values('TVT')
    typewell_gr_at_baseline = interpolate_with_extrapolation(
        typewell_sorted['TVT'].to_numpy(dtype=float),
        typewell_sorted['GR'].to_numpy(dtype=float),
        baseline_tvt,
    )
    nearest_typewell_tvt, nearest_typewell_gr_diff = nearest_typewell_by_gr(
        typewell, gr.to_numpy(dtype=float)
    )
    known_tvt = horizontal['TVT_input'].dropna()
    first_tvt = float(known_tvt.iloc[0]) if len(known_tvt) else np.nan
    last_tvt = float(known_tvt.iloc[-1]) if len(known_tvt) else np.nan
    features = pd.DataFrame(
        {
            'split': split,
            'well_id': well_id,
            'row_index': row_index,
            'n_rows': len(horizontal),
            'row_from_ps': row_index - ps_row,
            'frac_from_ps': (row_index - ps_row) / max(len(horizontal) - ps_row, 1),
            'MD': md,
            'md_from_ps': md - float(md.iloc[ps_row_safe]),
            'X': horizontal['X'],
            'Y': horizontal['Y'],
            'Z': horizontal['Z'],
            'x_from_ps': horizontal['X'] - float(horizontal['X'].iloc[ps_row_safe]),
            'y_from_ps': horizontal['Y'] - float(horizontal['Y'].iloc[ps_row_safe]),
            'z_from_ps': horizontal['Z'] - float(horizontal['Z'].iloc[ps_row_safe]),
            'gr': gr,
            'gr_was_missing': gr_raw.isna().astype(int),
            'gr_from_ps': gr - float(gr.iloc[ps_row_safe]),
            'gr_roll_mean_11': gr.rolling(11, center=True, min_periods=1).mean(),
            'gr_roll_mean_51': gr.rolling(51, center=True, min_periods=1).mean(),
            'gr_roll_std_51': gr.rolling(51, center=True, min_periods=2).std().fillna(0.0),
            'gr_delta_1': gr.diff(1).fillna(0.0),
            'gr_delta_10': gr.diff(10).fillna(0.0),
            'last_tvt_input': last_tvt,
            'first_tvt_input': first_tvt,
            'tvt_input_range': last_tvt - first_tvt,
            'tvt_slope_last_25': slope_last(horizontal['TVT_input'], md, 25),
            'tvt_slope_last_100': slope_last(horizontal['TVT_input'], md, 100),
            'baseline_tvt': baseline_tvt,
            'typewell_tvt_min': typewell['TVT'].min(),
            'typewell_tvt_max': typewell['TVT'].max(),
            'typewell_gr_at_baseline_tvt': typewell_gr_at_baseline,
            'gr_minus_typewell_gr_at_baseline': gr.to_numpy(dtype=float) - typewell_gr_at_baseline,
            'nearest_typewell_tvt_by_gr': nearest_typewell_tvt,
            'nearest_typewell_gr_diff': nearest_typewell_gr_diff,
        }
    )
    features['xy_dist_from_ps'] = np.sqrt(features['x_from_ps'] ** 2 + features['y_from_ps'] ** 2)
    features['is_prediction_row'] = tvt_input_missing.astype(int)
    if 'TVT' in horizontal.columns:
        features['target_tvt'] = horizontal['TVT']
    return features

def build_split(raw_dir, split):
    frames = []
    for horizontal_path in sorted((raw_dir / split).glob('*__horizontal_well.csv')):
        well_id = horizontal_path.name.split('__')[0]
        typewell_path = raw_dir / split / f'{well_id}__typewell.csv'
        frames.append(build_well_features(horizontal_path, typewell_path, split))
    return pd.concat(frames, ignore_index=True)

def choose_validation_wells(features, valid_size=VALID_SIZE, random_state=RANDOM_STATE):
    unique_wells = pd.Series(features['well_id'].unique())
    splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=random_state)
    dummy = np.zeros(len(unique_wells))
    groups = unique_wells.to_numpy()
    _, valid_idx = next(splitter.split(dummy, groups=groups))
    return set(unique_wells.iloc[valid_idx])

def sample_train_rows(train_features, valid_wells, max_rows=MAX_TRAIN_ROWS, chunksize=CHUNKSIZE, random_state=RANDOM_STATE):
    # Match src/train_model.py: read rows in chunks, keep training rows until max_rows,
    # and only sample inside the chunk that would exceed the limit.
    rng = np.random.default_rng(random_state)
    train_parts = []
    train_rows = 0
    for start in range(0, len(train_features), chunksize):
        chunk = train_features.iloc[start:start + chunksize]
        train_chunk = chunk.loc[~chunk['well_id'].isin(valid_wells)]
        remaining = max_rows - train_rows
        if remaining > 0 and len(train_chunk) > 0:
            if len(train_chunk) > remaining:
                selected = rng.choice(train_chunk.index.to_numpy(), size=remaining, replace=False)
                train_chunk = train_chunk.loc[selected]
            train_parts.append(train_chunk)
            train_rows += len(train_chunk)
    return pd.concat(train_parts, ignore_index=True)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('Building features...')
train_all = build_split(RAW_DIR, 'train')
test_all = build_split(RAW_DIR, 'test')
train_features = train_all[train_all['is_prediction_row'] == 1].copy()
test_features = test_all[test_all['is_prediction_row'] == 1].copy()
print('Train prediction rows:', len(train_features))
print('Test prediction rows:', len(test_features))

valid_wells = choose_validation_wells(train_features)
train_sample = sample_train_rows(train_features, valid_wells)
valid = train_features[train_features['well_id'].isin(valid_wells)].copy()
print('Train rows used:', len(train_sample))
print('Validation rows:', len(valid))
print('Validation wells:', len(valid_wells))

model = LGBMRegressor(
    objective='regression',
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.05,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=1,
)
model.fit(train_sample[FEATURE_COLUMNS], train_sample[TARGET_COLUMN])

valid_pred = model.predict(valid[FEATURE_COLUMNS])
baseline_pred = valid['baseline_tvt'].to_numpy(dtype=float)
print('CV model RMSE:', rmse(valid[TARGET_COLUMN].to_numpy(dtype=float), valid_pred))
print('CV baseline RMSE:', rmse(valid[TARGET_COLUMN].to_numpy(dtype=float), baseline_pred))

test_pred = model.predict(test_features[FEATURE_COLUMNS])
prediction = pd.DataFrame(
    {
        'id': test_features['well_id'].astype(str) + '_' + test_features['row_index'].astype(str),
        'tvt': test_pred,
    }
)
sample = pd.read_csv(RAW_DIR / 'sample_submission.csv')
submission = sample[['id']].merge(prediction, on='id', how='left')
assert not submission['tvt'].isna().any()
submission.to_csv(OUTPUT_PATH, index=False)
print('Wrote', OUTPUT_PATH)
print(submission.head())
